# FreeFlow sur notre vidéo : inférence et interprétation (Kaggle)

**Article :** V. Bargatin, A. Yakovenko, K. Abud, D. Vatolin — *FreeFlow: A Bias-free Hierarchical Transformer for Optical Flow Estimation*, ECCV 2026.
**Code et poids :** dépôt officiel `msu-video-group/freeflow`, poids Hugging Face `a-yakovenko`.

On utilise FreeFlow **tel quel**, avec les poids appris par les auteurs : **aucun réentraînement**, seulement de l'**inférence**. On donne des paires d'images, et le modèle rend le flot.

---

### Rappel en 5 lignes : ce que fait FreeFlow
1. **Patchification** : l'image ($3 \times H \times W$) est coupée en patchs de $8 \times 8$ pixels. Chaque patch devient une ligne de $D$ nombres, d'où une matrice $X$ de $N \times D$, avec $N = \frac{H}{8} \cdot \frac{W}{8}$.
2. **Encodeur siamois** : les deux images passent par le même encodeur ($L$ couches Win → Swin → Global), ce qui donne $F^1$ et $F^2$ ($N \times D$).
3. **Décodeur** : attention croisée. Chaque patch de l'image 1 cherche où il est dans l'image 2, ce qui donne $Z$ ($N \times D$).
4. **Tête** : $Z$ est transformé en $U$ ($5 \times H \times W$). Le flot est $f = 8 \cdot U_{[0:2]}$, c'est-à-dire $(u, v)$ pour chaque pixel.
5. Tout se fait en **une seule passe**, sans itérations (contrairement à RAFT).

### Ce qu'on mesure
| Mesure | Ce que ça dit |
|---|---|
| Tailles des matrices | si l'architecture correspond bien au papier |
| Temps par paire | la vitesse du modèle |
| Mémoire GPU | combien de mémoire de la carte graphique il utilise |
| Erreur photométrique | si le flot explique bien le changement entre deux images (notre vidéo n'a pas de vrai flot) |
| EPE, Fl-all (Middlebury) | la vraie précision, mesurée sur des images dont on connaît le flot exact |

## 1. Réglages

Les paramètres à choisir avant de lancer :

| Paramètre | Valeur choisie | Signification |
|---|---|---|
| `VIDEO_PATH` | notre vidéo | chemin de la vidéo ; si vide, le notebook prend le premier `.mp4` trouvé |
| `VARIANT` | `"L"` | taille du modèle : **S** (petit, rapide), **M** (moyen), **L** (grand, le plus précis) |
| `MAX_PAIRS` | `None` | nombre de paires à traiter ; `None` = toute la vidéo (un petit nombre sert à tester vite) |
| `MAX_SIDE` | `None` | réduit l'image si elle est trop grande ; `None` = taille d'origine |
| `CKPT` | `…FreeFlow-L-TaTSKH-HQ` | le **checkpoint**, c'est-à-dire le fichier des **poids appris** ; « TaTSKH-HQ » est la version générale recommandée par les auteurs |

In [ ]:
import glob, os
VIDEO_PATH = ""        # chemin d'un fichier .mp4, d'un dossier, ou "" = premier .mp4 trouvé dans /kaggle/input
VARIANT    = "L"       # "S", "M" ou "L"
MAX_PAIRS  = None      # None = toute la vidéo ; ex. 50 pour tester vite
MAX_SIDE   = None      # None = résolution d'origine ; ex. 1024 pour réduire

if not VIDEO_PATH or os.path.isdir(VIDEO_PATH):
    base = VIDEO_PATH or "/kaggle/input"
    found = sorted(glob.glob(f"{base}/**/*.mp4", recursive=True))
    assert found, f"Aucune vidéo .mp4 trouvée dans {base} : ajoute-la avec 'Add Input'."
    VIDEO_PATH = found[0]
CKPT = f"a-yakovenko/optical-flow-FreeFlow-{VARIANT}-TaTSKH-HQ"
OUT = "/kaggle/working/freeflow_out"
print("Vidéo :", VIDEO_PATH, "\nModèle :", CKPT)

### Interprétation
La vidéo `video voiture 7s.mp4` a été trouvée, et on utilise **FreeFlow-L** (la plus grande variante, la plus précise), avec le checkpoint général **TaTSKH-HQ**.

## 2. Installation du code officiel

- `git clone` télécharge le code des auteurs depuis GitHub.
- `pip install` l'installe, avec les bibliothèques dont il a besoin.


In [ ]:
import os, sys
REPO = "/kaggle/working/freeflow_repo"
if not os.path.exists(REPO):
    !git clone -q https://github.com/msu-video-group/freeflow.git {REPO}
!pip install -q {REPO}

## 3. Chargement du modèle

- `FreeFlow.from_pretrained(CKPT)` télécharge les **poids appris** (environ 0,9 Go pour FreeFlow-L) et construit le réseau.
- `.eval()` met le modèle en mode **inférence** : on l'utilise, on ne l'entraîne pas.
- `.to("cuda")` place le modèle sur la **carte graphique (GPU)**, beaucoup plus rapide que le processeur.
- **fp16** : on calcule en **demi-précision** (nombres sur 16 bits au lieu de 32). C'est plus rapide et ça prend moins de mémoire. La T4 n'a pas de bf16 matériel, d'où fp16.

On affiche ensuite la **configuration** du modèle pour la comparer au papier (Tab. 1).

In [ ]:
import time, numpy as np, torch, torch.nn.functional as F, cv2, matplotlib.pyplot as plt
from freeflow import FreeFlow
from freeflow.utils.visualization import flow_to_image
import freeflow.models.blocks as ffblocks

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU"
AMP = torch.float16 if DEVICE == "cuda" else None   # T4 / P100 : pas de bf16 matériel -> fp16
print("GPU :", GPU)

t0 = time.time()
model = FreeFlow.from_pretrained(CKPT).eval().to(DEVICE)
print(f"Chargé en {time.time()-t0:.1f} s")

a = model.enc_blocks[0].block1.attn
cfg = {"couches L": model.enc_depth, "dimension D": model.enc_embed_dim, "têtes h": a.num_heads,
       "dimension par tête d": model.enc_embed_dim // a.num_heads, "patch": tuple(model.patch_embed.patch_size),
       "fenêtres": tuple(model.window_size), "réduction Global": model.down_ratio,
       "paramètres": f"{sum(p.numel() for p in model.parameters())/1e6:.1f} M"}
for k, v in cfg.items(): print(f"{k:22s}: {v}")

### Interprétation : la configuration est celle du papier

| Paramètre affiché | Valeur | Signification |
|---|---|---|
| couches $L$ | 8 | on répète 8 fois Win → Swin → Global (dans l'encodeur et dans le décodeur) |
| dimension $D$ | 512 | nombre de valeurs qui décrivent chaque patch |
| têtes $h$ | 8 | 8 attentions en parallèle dans chaque bloc |
| dimension par tête $d$ | 64 | $512 / 8 = 64$ |
| patch | (8, 8) | chaque patch fait 8 × 8 pixels |
| fenêtres | (4, 4) | l'image est découpée en $4 \times 4 = 16$ fenêtres (Win, Swin) |
| réduction Global | 2 | le Global travaille sur une image 2 fois plus petite dans chaque sens |
| paramètres | 230,7 M | **230,7 millions** de nombres appris : identique au papier pour FreeFlow-L |

Le modèle a été chargé sur le GPU **Tesla T4** en 3,2 s.

## 4. Lecture de la vidéo

On lit la vidéo image par image avec OpenCV. Une vidéo de $T$ images donne $T - 1$ **paires** $(I_t, I_{t+1})$. FreeFlow calcule un flot pour chaque paire, **indépendamment** des autres (pas de mémoire du passé, contrairement à MemFlow).

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
FPS = cap.get(cv2.CAP_PROP_FPS)
frames = []
while True:
    ok, fr = cap.read()
    if not ok: break
    fr = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)
    if MAX_SIDE and max(fr.shape[:2]) > MAX_SIDE:
        s = MAX_SIDE / max(fr.shape[:2])
        fr = cv2.resize(fr, (round(fr.shape[1]*s)//2*2, round(fr.shape[0]*s)//2*2), interpolation=cv2.INTER_AREA)
    frames.append(fr)
cap.release()
H, W = frames[0].shape[:2]
n_pairs = len(frames) - 1 if MAX_PAIRS is None else min(MAX_PAIRS, len(frames) - 1)
print(f"{len(frames)} images de {H}×{W} (H×W) à {FPS:.0f} fps → {n_pairs} paires (Iₜ, Iₜ₊₁)")

### Interprétation
La vidéo contient **212 images** de **1024 × 576** pixels (hauteur × largeur, format portrait) à **30 images par seconde**, soit environ 7 s. On obtient **211 paires**, donc 211 flots à calculer.

*Remarque : les notebooks RAFT et MemFlow, qui lisent la vidéo avec ffmpeg, trouvaient 210 images. C'est la même vidéo, seul l'outil de lecture diffère. L'écart est négligeable (erreur « sans flot » 1,20 ici au lieu de 1,22).*

## 5. Voir l'architecture en action : les tailles des matrices

On fait passer **une seule paire** dans le réseau, et on « espionne » la taille des matrices à chaque étape. On compare avec les formules de la présentation :

$$N = \frac{H}{8} \cdot \frac{W}{8}, \qquad \text{fenêtre} = \frac{N}{16}, \qquad \text{Global} = \frac{N}{4}$$

(Si $H$ ou $W$ n'est pas un multiple de 32, le modèle complète l'image jusqu'au multiple suivant. Ce n'est pas le cas ici : 1024 et 576 sont des multiples de 32.)

In [ ]:
def to_input(a, b):
    x = torch.from_numpy(np.stack([a, b])).permute(0, 3, 1, 2)[None].float()   # [1, 2, 3, H, W] dans [0, 255]
    return x.to(DEVICE)

def amp():
    return torch.autocast("cuda", dtype=AMP) if AMP else torch.autocast("cpu", enabled=False)

@torch.inference_mode()
def predict(a, b):
    with amp():
        return model(to_input(a, b)).float()[0]      # [2, H, W] : u et v en pixels

Hp, Wp = -(-H // 32) * 32, -(-W // 32) * 32
N, D = (Hp // 8) * (Wp // 8), model.enc_embed_dim
print(f"Image complétée : {Hp}×{Wp}   →   N = {Hp//8}×{Wp//8} = {N}   D = {D}")
print(f"Prévu : fenêtre N/16 = {N//16} tokens   ·   Global N/4 = {N//4} tokens\n")

log = []
shape = lambda t: tuple(t.shape) if torch.is_tensor(t) else (shape(t[0]) if isinstance(t, (tuple, list)) else None)
def hook(name):
    return lambda m, i, o: log.append((name, shape(o)))
hooks = [model.patch_embed.register_forward_hook(hook("1. patchification  → X")),
         model.enc_norm.register_forward_hook(hook("2. sortie encodeur → F¹, F² (lot de 2)")),
         model.dec_norm.register_forward_hook(hook("3. sortie décodeur → Z")),
         model.head.register_forward_hook(hook("4. tête            → U (5 canaux)"))]
_sdpa = torch.nn.functional.scaled_dot_product_attention
att = []
def spy(q, k, v, *args, **kw):
    if len(att) < 400: att.append((tuple(q.shape), tuple(k.shape)))
    return _sdpa(q, k, v, *args, **kw)
ffblocks.scaled_dot_product_attention = spy
flow = predict(frames[0], frames[1])
ffblocks.scaled_dot_product_attention = _sdpa
for h_ in hooks: h_.remove()

for name, s in log: print(f"{name:42s} {s}")
print(f"{'5. flot final      → f = 8·U[0:2]':42s} {tuple(flow.shape)}\n")
print("Attentions rencontrées  (Q : lots × têtes × tokens × 64) :")
for (qs, ks) in dict.fromkeys(att):
    kind = "fenêtres (Win/Swin)" if qs[2] == N // 16 else "Global" if qs[2] == N // 4 else "?"
    print(f"   Q {qs}   K {ks}   → {qs[2]} tokens = {kind}")

### Interprétation : l'architecture est exactement celle du papier

| Étape | Taille mesurée | Explication |
|---|---|---|
| Patchification → $X$ | (2, 9216, 512) | **2** images traitées ensemble (encodeur siamois), **9216** patchs ($128 \times 72$), **512** nombres par patch ($D$) |
| Sortie encodeur → $F^1, F^2$ | (2, 9216, 512) | même taille : l'encodeur enrichit les descriptions sans changer la forme |
| Sortie décodeur → $Z$ | (1, 9216, 512) | **1** seule matrice : une ligne par patch de l'image 1 (les *flow tokens*) |
| Tête → $U$ | (1, 5, 1024, 576) | **5** canaux par pixel, à la taille de l'image |
| Flot final → $f$ | (2, 1024, 576) | les **2** grilles $u$ et $v$ |

**Les attentions** : la forme de $Q$ est (lots, têtes, tokens, 64).
- (32, 8, **576**, 64) : **Win / Swin de l'encodeur**. 32 = 2 images × 16 fenêtres, et 576 = $\frac{N}{16}$ patchs par fenêtre.
- (2, 8, **2304**, 64) : **Global de l'encodeur**. 2304 = $\frac{N}{4}$ tokens (carte réduite par 2).
- (16, 8, 576, 64) et (1, 8, 2304, 64) : les mêmes dans le **décodeur**, avec une seule image qui pose les questions (l'image 1).
- **8** têtes de **64** dimensions : $8 \times 64 = 512 = D$.

**Conclusion :** toutes les tailles correspondent aux formules. Le code fait exactement ce qui est décrit dans le papier et dans notre présentation.

## 6. Flot sur toute la vidéo : temps, mémoire, erreur

Pour chaque paire $(I_t, I_{t+1})$, la cellule :
1. **calcule le flot** $f = (u, v)$ avec FreeFlow ;
2. **chronomètre** le calcul (temps par paire) ;
3. **mesure l'erreur photométrique** : on déplace l'image $I_{t+1}$ selon le flot pour essayer de retrouver $I_t$, puis on calcule la différence moyenne
$$\text{erreur} = \text{moyenne}\ \big|\, I_t(x, y) - I_{t+1}(x + u,\ y + v) \,\big|$$
   On la compare à l'erreur **sans flot**, $|I_t - I_{t+1}|$. Plus l'erreur avec flot est basse, mieux le flot explique le mouvement.

À la fin, on affiche le **pic de mémoire GPU** : la mémoire maximale utilisée pendant tout le calcul.

In [ ]:
os.makedirs(OUT, exist_ok=True)
if DEVICE == "cuda":
    torch.cuda.reset_peak_memory_stats()
times, mags, photo, photo0 = [], [], [], []

def warp_error(a, b, f):
    # |Iₜ − Iₜ₊₁(x+u, y+v)| : erreur photométrique (pas de vérité terrain sur notre vidéo)
    g0, g1 = [cv2.cvtColor(x, cv2.COLOR_RGB2GRAY).astype(np.float32) for x in (a, b)]
    gx, gy = np.meshgrid(np.arange(W, dtype=np.float32), np.arange(H, dtype=np.float32))
    w = cv2.remap(g1, gx + f[0], gy + f[1], cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
    return float(np.abs(g0 - w).mean()), float(np.abs(g0 - g1).mean())

flows_png = []
KEEP = sorted(set(np.linspace(0, n_pairs - 1, 4).astype(int)))   # 4 instants gardés pour les figures
kept = {}
for t in range(n_pairs):
    if DEVICE == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    f = predict(frames[t], frames[t + 1])
    if DEVICE == "cuda": torch.cuda.synchronize()
    times.append(1000 * (time.perf_counter() - t0))
    f = f.cpu().numpy()
    mags.append(np.linalg.norm(f, axis=0).mean())
    e, e0 = warp_error(frames[t], frames[t + 1], f); photo.append(e); photo0.append(e0)
    flows_png.append(flow_to_image(f.transpose(1, 2, 0), None))
    if t in KEEP: kept[t] = f
    if t == n_pairs // 2: f_mid = f

steady = np.array(times[3:] if len(times) > 5 else times)
mem = torch.cuda.max_memory_allocated() / 2**20 if DEVICE == "cuda" else float("nan")
print(f"FreeFlow-{VARIANT} | {GPU} | {W}×{H} | {n_pairs} paires")
print(f"Temps par paire     : {steady.mean():.0f} ms  ({1000/steady.mean():.2f} paires/s)")
print(f"Pic mémoire GPU     : {mem:.0f} Mo")
print(f"Erreur photométrique: {np.mean(photo):.2f} avec le flot  vs  {np.mean(photo0):.2f} sans flot")

### Interprétation

| Mesure | Résultat | Ce que ça veut dire |
|---|---|---|
| **Temps par paire** | **573 ms** (1,75 paire/s) | rapide pour un modèle de 231 M paramètres ; **plus rapide que RAFT** sur la même T4 (705 ms dans notre notebook RAFT) |
| **Pic mémoire GPU** | **1367 Mo** | faible : FreeFlow ne stocke jamais de grande matrice $N \times N$ (fenêtres + Global réduit), contrairement au volume de corrélation de RAFT |
| **Erreur photométrique** | **0,49** avec flot vs **1,20** sans flot | le flot divise l'erreur par plus de 2 : il explique bien le mouvement |

**Pourquoi l'erreur n'est pas à 0 :** une partie du changement entre deux images ne peut pas s'expliquer par un mouvement : les **occlusions** (le fond qui apparaît derrière la voiture), les **reflets**, le **flou de mouvement** et la **compression** de la vidéo.

## 7. Résultats visuels

**Figure 1, pour la paire du milieu de la vidéo :**
1. **l'image** ;
2. **le flot en couleurs** : la **couleur** indique la **direction** du mouvement, l'**intensité** de la couleur sa **vitesse**, et le **blanc** veut dire immobile ;
3. **la magnitude** $|f| = \sqrt{u^2 + v^2}$ : de combien de pixels chaque pixel a bougé (noir = 0) ;
4. **les vecteurs** $(u, v)$ : de petites flèches jaunes qui montrent le sens du mouvement.

**Figure 2, au fil de la vidéo :** le temps par paire, le mouvement moyen, et l'erreur photométrique avec et sans flot.

In [ ]:
t = n_pairs // 2
mag = np.linalg.norm(f_mid, axis=0)
fig, ax = plt.subplots(1, 4, figsize=(20, 6))
ax[0].imshow(frames[t]); ax[0].set_title(f"Image {t}")
ax[1].imshow(flows_png[t]); ax[1].set_title("Flot (couleur = direction, intensité = vitesse)")
im = ax[2].imshow(mag, cmap="inferno"); ax[2].set_title("|f| = √(u²+v²) en pixels"); plt.colorbar(im, ax=ax[2], fraction=0.03)
step = max(8, W // 30); ys, xs = np.mgrid[step//2:H:step, step//2:W:step]
ax[3].imshow(frames[t]); ax[3].quiver(xs, ys, f_mid[0, ys, xs], f_mid[1, ys, xs], color="yellow", angles="xy", scale_units="xy", scale=0.5)
ax[3].set_title("Vecteurs (u, v)")
for a_ in ax: a_.axis("off")
plt.tight_layout(); plt.savefig(f"{OUT}/resultat.png", dpi=120); plt.show()

y, x = np.unravel_index(np.argmax(mag), mag.shape)
print(f"Exemple : le pixel (x={x}, y={y}) a u={f_mid[0,y,x]:.1f}, v={f_mid[1,y,x]:.1f} "
      f"→ il se retrouve en ({x+f_mid[0,y,x]:.0f}, {y+f_mid[1,y,x]:.0f}) dans l'image suivante.")

### Interprétation de la figure 1 (paire du milieu)
- **Le fond est blanc** dans le flot et **noir** dans la magnitude : il ne bouge pas. La caméra est fixe.
- **La voiture est d'une seule couleur** (violet) : tous ses pixels bougent dans **la même direction**. C'est un **mouvement rigide**.
- **Magnitude ≈ 5 px, uniforme** sur toute la voiture : elle avance d'un bloc, sans tourner.
- **Les contours sont nets**, jusqu'aux petits rétroviseurs : la tête de FreeFlow donne une valeur pour chaque pixel.

**L'exemple chiffré :** le pixel $(x = 360,\ y = 568)$ a $u = 0{,}4$ (presque pas de mouvement horizontal) et $v = -5{,}5$ ($v$ négatif = vers le haut). Il se retrouve en $(360,\ 563)$ dans l'image suivante. **La voiture avance tout droit vers le haut**, à environ $5{,}5 \times 30 \approx 165$ pixels par seconde.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 3.5))
ax[0].plot(times); ax[0].set_title("Temps par paire (ms)"); ax[0].set_xlabel("paire")
ax[1].plot(mags); ax[1].set_title("Mouvement moyen |f| (px)"); ax[1].set_xlabel("paire")
ax[2].plot(photo0, label="sans flot"); ax[2].plot(photo, label="avec FreeFlow"); ax[2].legend()
ax[2].set_title("Erreur photométrique (plus bas = mieux)"); ax[2].set_xlabel("paire")
plt.tight_layout(); plt.savefig(f"{OUT}/courbes.png", dpi=120); plt.show()

### Interprétation de la figure 2 (au fil de la vidéo)
- **Temps par paire :** entre environ 530 et 600 ms. Il **augmente légèrement** au fil de la vidéo alors que FreeFlow fait toujours le même calcul (une passe, sans itérations) : la hausse vient du GPU (température, fréquence), pas du modèle.
- **Mouvement moyen :** proche de 0 au début (voiture arrêtée), puis il augmente avec des pics vers les paires 140 à 190 (la voiture va plus vite), puis il retombe à 0 (la voiture est sortie de l'image).
- **Erreur photométrique :** la courbe **orange** (avec FreeFlow) reste **toujours sous** la bleue (sans flot). Les pics d'erreur coïncident avec les moments rapides : flou de mouvement et occlusions plus importantes. C'est une difficulté connue de tous les modèles de flot.

## 8. Tableau des métriques sur notre vidéo

Le résumé des mesures de la partie 6. La **réduction de l'erreur** est le pourcentage de la différence entre deux images que le flot explique :
$$\text{réduction} = 1 - \frac{\text{erreur avec flot}}{\text{erreur sans flot}}$$

In [ ]:
import pandas as pd
red = 100 * (1 - np.mean(photo) / np.mean(photo0))
display(pd.DataFrame({
    "Métrique": ["Temps par paire", "Pic mémoire GPU", "Erreur photométrique avec flot", "Erreur photométrique sans flot", "Réduction de l'erreur"],
    "Valeur": [f"{steady.mean():.0f} ms", f"{mem:.0f} Mo", f"{np.mean(photo):.2f}", f"{np.mean(photo0):.2f}", f"{red:.0f} %"]}))

### Interprétation
Le flot **explique 59 %** de la différence entre deux images consécutives, avec seulement 1,4 Go de mémoire et un peu plus d'une demi-seconde par paire. Ce sont de bons résultats pour un modèle utilisé **sans réentraînement** sur une vidéo qu'il n'a jamais vue.

## 9. Quatre instants de la vidéo

Début, premier tiers, deuxième tiers et fin de la vidéo. Pour chaque instant : l'image, le flot en couleurs et la magnitude. Les **magnitudes** utilisent la **même échelle** de couleurs, pour pouvoir comparer les vitesses entre instants.

In [ ]:
vmax = max(np.linalg.norm(kept[t], axis=0).max() for t in kept)
fig, ax = plt.subplots(3, len(kept), figsize=(4.5 * len(kept), 12))
for j, t in enumerate(kept):
    ax[0, j].imshow(frames[t]); ax[0, j].set_title(f"Image {t}")
    ax[1, j].imshow(flows_png[t]); ax[1, j].set_title(f"Flot {t} → {t+1}")
    ax[2, j].imshow(np.linalg.norm(kept[t], axis=0), cmap="inferno", vmin=0, vmax=vmax); ax[2, j].set_title("|f| (px)")
for a_ in ax.ravel(): a_.axis("off")
plt.tight_layout(); plt.savefig(f"{OUT}/instants.png", dpi=110); plt.show()

###  Interprétation
- **Images 70 et 140 :** le fond est blanc (immobile) et la voiture violette. La **magnitude** est faible à l'image 70 (≈ 1 px, violet foncé) et forte à l'image 140 (≈ 5 px, jaune) : **la voiture accélère**.
- **Images 0 et 210 : le fond paraît coloré, mais ce n'est pas une erreur.** La voiture est **arrêtée** (image 0) ou **sortie de l'image** (image 210), donc le flot vaut presque 0 partout : la **magnitude est entièrement noire**. La couleur vient du fait que chaque image de flot est **normalisée par son propre maximum**. Quand tout est presque à 0, de minuscules variations (environ 0,1 px) deviennent visibles en couleur.

## 10. Précision avec vérité terrain : Middlebury

Sur notre vidéo, on ne connaît pas le **vrai** flot, donc on ne peut pas calculer l'erreur exacte. On utilise donc **Middlebury**, un jeu de données classique de 8 séquences où le vrai flot est connu (on prend la paire frame10 → frame11 de chaque séquence). C'est le même protocole que pour RAFT et MemFlow.

| Métrique | Définition | Meilleur |
|---|---|---|
| **EPE** (*End-Point Error*) | distance moyenne, en pixels, entre la pointe de la flèche prédite et celle de la vraie flèche : $\text{EPE} = \text{moyenne}\ \|f_{\text{prédit}} - f_{\text{vrai}}\|$ | plus bas |
| **Fl-all** | % de pixels « faux » : erreur > 3 px **et** > 5 % du vrai mouvement | plus bas |
| **EPE flot nul** | l'EPE qu'on aurait en prédisant « rien ne bouge » ; sert de **référence** | — |

In [ ]:
import subprocess
def read_flo(p):
    with open(p, "rb") as fh:
        np.fromfile(fh, np.float32, 1); w, h = np.fromfile(fh, np.int32, 2)
        return np.fromfile(fh, np.float32, 2 * w * h).reshape(h, w, 2)
MB = "/kaggle/working/middlebury"
seqs = ["Dimetrodon", "Grove2", "Grove3", "Hydrangea", "RubberWhale", "Urban2", "Urban3", "Venus"]
rows, preds = [], {}
try:
    if not os.path.exists(f"{MB}/other-gt-flow"):
        os.makedirs(MB, exist_ok=True)
        for z in ["other-gt-flow.zip", "other-color-twoframes.zip"]:
            subprocess.run(["wget", "-q", "-O", f"{MB}/{z}", f"https://vision.middlebury.edu/flow/data/comp/zip/{z}"], check=True)
            subprocess.run(["unzip", "-q", "-o", f"{MB}/{z}", "-d", MB], check=True)
    for sname in seqs:
        a, b = [cv2.cvtColor(cv2.imread(f"{MB}/other-data/{sname}/frame{k}.png"), cv2.COLOR_BGR2RGB) for k in (10, 11)]
        gt = read_flo(f"{MB}/other-gt-flow/{sname}/flow10.flo"); valid = (np.abs(gt) < 1e9).all(-1)
        pr = predict(a, b).cpu().numpy().transpose(1, 2, 0); preds[sname] = (pr, gt, valid)
        e = np.linalg.norm(pr - gt, axis=-1)[valid]; gm = np.linalg.norm(gt, axis=-1)[valid]
        rows.append({"séquence": sname, "EPE (px)": e.mean(), "Fl-all (%)": 100 * ((e > 3) & (e > 0.05 * gm)).mean(),
                     "EPE flot nul (px)": gm.mean()})
    mb = pd.DataFrame(rows).set_index("séquence"); mb.loc["moyenne"] = mb.mean()
    display(mb.round(3))
    fig, ax = plt.subplots(2, 4, figsize=(20, 7))
    for a_, sname in zip(ax.ravel(), seqs):
        pr, gt, valid = preds[sname]; g = gt.copy(); g[~valid] = 0
        a_.imshow(np.hstack([flow_to_image(g, None), flow_to_image(pr, None)]))
        a_.set_title(f"{sname} : vrai | FreeFlow"); a_.axis("off")
    plt.tight_layout(); plt.savefig(f"{OUT}/middlebury.png", dpi=100); plt.show()
except Exception as ex:
    print("Middlebury indisponible (Internet coupé ou site lent) :", ex)

### Interprétation : précision très bonne

- **EPE moyenne = 0,308 px**, contre **4,19 px** pour un flot nul. L'erreur est **environ 13 fois plus petite**. En moyenne, la flèche prédite est à moins d'un tiers de pixel de la vraie.
- **Fl-all = 1,12 %** : seulement environ 1 pixel sur 100 est considéré comme faux.
- **Meilleures séquences :** Grove2, Urban2, Urban3 (EPE ≈ 0,15 px), avec des textures riches et des mouvements nets.
- **Séquences plus difficiles :** Dimetrodon, RubberWhale (EPE ≈ 0,5 px), avec de très petits mouvements et des zones sans texture. C'est **difficile pour tous les modèles**.

**Attention à la figure :** certaines couleurs diffèrent entre « vrai » et « FreeFlow » (Dimetrodon, Hydrangea, RubberWhale). C'est **normal** : chaque panneau a sa **propre échelle de couleurs**. Les **formes** sont les mêmes, et c'est l'**EPE** (dans le tableau) qui mesure la vraie erreur.

**Remarque :** ce n'est pas le benchmark du papier (Sintel, KITTI, Spring). Middlebury sert à montrer de vraies métriques avec vérité terrain, avec le même protocole que pour RAFT et MemFlow.

## 11. Vidéo résultat

On crée une vidéo avec, côte à côte, l'image d'origine et son flot en couleurs. Elle se télécharge dans l'onglet **Output** (dossier `freeflow_out`), avec toutes les figures en `.png`.

In [ ]:
path = f"{OUT}/freeflow_video.mp4"
vw = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*"mp4v"), FPS, (2 * W, H))
for t in range(n_pairs):
    vw.write(cv2.cvtColor(np.hstack([frames[t], flows_png[t]]), cv2.COLOR_RGB2BGR))
vw.release()
print("Vidéo enregistrée :", path, "(onglet Output à droite pour la télécharger)")

## Résumé : les chiffres à recopier dans la présentation

In [ ]:
print(f"Modèle               : FreeFlow-{VARIANT} TaTSKH-HQ (fp16)")
print(f"GPU                  : {GPU}")
print(f"Vidéo                : {W}×{H}, {FPS:.0f} fps, {n_pairs} paires")
print(f"Tailles              : N = {N}, D = {D}, fenêtre = {N//16}, Global = {N//4}")
print(f"Temps par paire      : {steady.mean():.0f} ms")
print(f"Pic mémoire GPU      : {mem:.0f} Mo")
print(f"Erreur photométrique : {np.mean(photo):.2f} (sans flot : {np.mean(photo0):.2f}, réduction {red:.0f} %)")
if rows: print(f"Middlebury           : EPE {mb.loc['moyenne','EPE (px)']:.3f} px · Fl-all {mb.loc['moyenne','Fl-all (%)']:.2f} %")

###  Conclusion

| Résultat | Valeur | Verdict |
|---|---|---|
| Architecture (tailles, paramètres) | $N = 9216$, $D = 512$, fenêtre 576, Global 2304, 230,7 M | ✅ identique au papier |
| Temps par paire | 573 ms | ✅ bon, plus rapide que RAFT sur la même T4 |
| Mémoire GPU | 1367 Mo | ✅ faible |
| Erreur photométrique | 0,49 vs 1,20 (−59 %) | ✅ le flot explique bien le mouvement |
| Middlebury | EPE 0,308 px · Fl-all 1,12 % | ✅ très précis |
| Moments rapides, séquences sans texture | erreur un peu plus haute | ⚠️ cas difficiles connus, normal |

**Conclusion:**
> « Sans réentraînement, FreeFlow-L traite une paire 1024 × 576 en 573 ms sur une T4, avec 1,4 Go de mémoire. Le flot réduit l'erreur photométrique de 59 %, et sur Middlebury, où l'on connaît le vrai flot, l'erreur moyenne est de 0,31 pixel. Sur notre vidéo, le fond a un flot nul et la voiture un mouvement uniforme vers le haut, de plus en plus rapide. »